## Libraries & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import joblib
file_name = 'Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet' 
print("⏳ Loading Parquet Dataset (Fast Mode)...")
df = pd.read_parquet(file_name)
print("Columns in Dataset:", df.columns.tolist())
df.head()
print(df.columns.tolist())

## Data Cleaning & Pre-processing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
features = [
    'Protocol', 
    'Flow Duration', 
    'Total Fwd Packets', 
    'Total Backward Packets', 
    'Flow IAT Mean', 
    'Fwd Packet Length Max', 
    'Fwd Packet Length Min', 
    'Packet Length Mean' 
]
X = df[features]
y = df['Label']
le = LabelEncoder()
y = le.fit_transform(y)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print("✅ Step 2: Pre-processing Complete!")
print(f"Total Rows: {len(df)} | Training Rows: {len(X_train)}")

## Model Training & Saving

In [ ]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
xgb = XGBClassifier(n_estimators=50, learning_rate=0.1, random_state=42)
smart_shield = VotingClassifier(estimators=[('rf', rf), ('xgb', xgb)], voting='soft')
print("🧠 Training Smart Shield Hybrid Model... Please wait.")
smart_shield.fit(X_train, y_train)
joblib.dump(smart_shield, 'smart_shield_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("✅ Model Trained and Saved Successfully!")

## Dashboard

In [ ]:
import matplotlib.pyplot as plt
label_counts = df['Label'].value_counts()
plt.figure(figsize=(8, 6))
colors = ['#66b3ff', '#ff9999'] 
explode = (0, 0.1)  
plt.pie(label_counts, 
        labels=label_counts.index, 
        autopct='%1.1f%%', 
        startangle=140, 
        colors=colors, 
        explode=explode, 
        shadow=True)
plt.title('Distribution of Network Traffic (Benign vs Bot)', fontsize=14)
plt.axis('equal') 
plt.show()
print("Traffic Counts:")
print(label_counts)

## Confusion Matrix Code (Using Seaborn)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
y_pred = smart_shield.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, 
            yticklabels=le.classes_)
plt.title('🛡️ Smart Shield: Performance Matrix', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()
print(classification_report(y_test, y_pred, target_names=le.classes_))